<a href="https://colab.research.google.com/github/2024meb1326-eng/ENSO-Climate-Forecasting-Model/blob/main/notebooks/forecasting_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import datetime
import tensorflow as tf
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, GlobalMaxPooling1D, LSTM, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [ ]:
def load_data():
  df = pd.read_csv("merged_enso.csv", parse_dates=["Date"])
def create_feature_for_date(target_date, df, feature_cols):
    if isinstance(target_date, str):
        target_date = pd.to_datetime(target_date)
    elif isinstance(target_date, datetime.date):
        target_date = pd.Timestamp(target_date)

    df_sorted = df.sort_values('Date')
    time_diffs = (df_sorted['Date'] - target_date).dt.days.abs()
    closest_idx = time_diffs.idxmin()
    closest_position = df_sorted.index.get_loc(closest_idx)

    month = target_date.month
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)

    start_idx = max(0, closest_position - 12)
    end_idx = closest_position + 1
    recent_data = df_sorted.iloc[start_idx:end_idx]

    if len(recent_data) == 0:
        feature_values = df[feature_cols].mean()
    else:
        feature_values = recent_data[feature_cols].mean()

    if 'month_sin' in feature_cols:
        feature_values['month_sin'] = month_sin
    if 'month_cos' in feature_cols:
        feature_values['month_cos'] = month_cos

    return feature_values.values.reshape(1, -1)


In [ ]:
data_loaded = True
try:
    df = load_data()
    if df is None:
        data_loaded = False
    else:
        sst_ds = load_sst_dataset()
        df = df.sort_values("Date").reset_index(drop=True)
        split_idx = int(0.8 * len(df))

        feature_cols = [
            "SST_Anomaly", "SOI", "SOI_lag_1", "SOI_lag_2", "SOI_lag_3",
            "SST_Anomaly_lag_1", "SST_Anomaly_lag_2", "SST_Anomaly_lag_3",
            "month_sin", "month_cos"
        ]
        X = df[feature_cols]
        y = df["ENSO_Label"]

        X_train = X.iloc[:split_idx]
        y_train = y.iloc[:split_idx]
        X_test = X.iloc[split_idx:]
        y_test = y.iloc[split_idx:]

        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        training_data_size = len(y_train)
        total_predictions = len(y_test)
        correct_predictions = int(accuracy * total_predictions)

        label_map = {0: "La Niña", 1: "Neutral", 2: "El Niño"}

        df.loc[df.index[split_idx:], "Predicted_Phase"] = [label_map[i] for i in y_pred]

        df.loc[df.index[split_idx:], "True_Phase"] = [label_map[i] for i in y_test]

In [ ]:
    cm = confusion_matrix(df["True_Phase"], df["Predicted_Phase"], labels=["El Niño", "Neutral", "La Niña"])
    report = classification_report(df["True_Phase"], df["Predicted_Phase"], output_dict=True)
if hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            "Feature": feature_cols,
            "Importance": model.feature_importances_
        }).sort_values("Importance", ascending=True)

In [ ]:
filtered_df = filtered_df.sort_values("Date").reset_index(drop=True)

            feature_cols = [
                "SST_Anomaly", "SOI", "SOI_lag_1", "SOI_lag_2", "SOI_lag_3",
                "SST_Anomaly_lag_1", "SST_Anomaly_lag_2", "SST_Anomaly_lag_3",
                "month_sin", "month_cos"
            ]

            enhanced_features = [
                "SST_Anomaly_rolling_3m", "SOI_rolling_3m", "SST_Anomaly_rolling_6m",
                "SST_SOI_interaction", "SST_trend", "SOI_trend", "is_djf", "is_jja"
            ]

            for feat in enhanced_features:
                if feat in filtered_df.columns and not filtered_df[feat].isna().all():
                    feature_cols.append(feat)

            if "OHC_Anomaly" in filtered_df.columns and not filtered_df["OHC_Anomaly"].isna().all():
                feature_cols.append("OHC_Anomaly")

In [ ]:
            X_custom = filtered_df[feature_cols].fillna(method='ffill').fillna(method='bfill')
            y_custom = filtered_df["ENSO_Label"]

            split_idx = int((100 - test_size) / 100 * len(filtered_df))
            split_idx = max(split_idx, int(0.6 * len(filtered_df)))

            X_train_custom = X_custom.iloc[:split_idx]
            y_train_custom = y_custom.iloc[:split_idx]
            X_test_custom = X_custom.iloc[split_idx:]
            y_test_custom = y_custom.iloc[split_idx:]
             scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_custom)
            X_test_scaled = scaler.transform(X_test_custom)